![image](car.jpeg)

**Car-ing is sharing**, an auto dealership company for car sales and rental, is taking their services to the next level thanks to **Large Language Models (LLMs)**.

As their newly recruited AI and NLP developer, you've been asked to prototype a chatbot app with multiple functionalities that not only assist customers but also provide support to human agents in the company.

The solution should receive textual prompts and use a variety of pre-trained Hugging Face LLMs to respond to a series of tasks, e.g. classifying the sentiment in a car’s text review, answering a customer question, summarizing or translating text, etc.


**TASK 1: Classify car reviews**

In [148]:
import pandas as pd
from transformers import pipeline
from sklearn.metrics import accuracy_score, f1_score

In [149]:
import os

# This will scan your entire workspace and print out where the file is located
for root, dirs, files in os.walk("."):
    for file in files:
        if "car_reviews" in file:
            print("Found it here:", os.path.join(root, file))

Found it here: ./data/car_reviews.csv


In [150]:
# 1. Load the dataset using the correct path
df = pd.read_csv('data/car_reviews.csv', delimiter=';')

# Extract text reviews and real labels into lists
reviews = df['Review'].tolist()
real_labels = df['Class'].tolist()



In [151]:
# 2. Load the pre-trained sentiment analysis model
classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

Device set to use cpu


In [152]:
# 3. Store the raw model outputs in predicted_labels
predicted_labels = classifier(reviews)

In [153]:
# 4. Map the model outputs ('POSITIVE' or 'NEGATIVE') onto a list of {0,1} binary labels called predictions
predictions = [1 if result['label'] == 'POSITIVE' else 0 for result in predicted_labels]

In [154]:
# 5. Map the ground-truth real_labels to {0,1} binary format for comparison
references = [1 if label == 'POSITIVE' else 0 for label in real_labels]

In [155]:
# 6. Calculate the metrics using the 'evaluate' library 
accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

accuracy_result = accuracy_metric.compute(references=references, predictions=predictions)['accuracy']
f1_result = f1_metric.compute(references=references, predictions=predictions)['f1']

In [156]:
# Print results to verify
print("Predictions:", predictions)
print("Accuracy Result:", accuracy_result)
print("F1 Result:", f1_result)

Predictions: [1, 1, 1, 0, 1]
Accuracy Result: 0.8
F1 Result: 0.8571428571428571


**TASK 2: Translate a car **review******

In [157]:
# 1. Grab the full first review (index 0) 
first_review = df['Review'].iloc[0]

In [158]:
# 2. Load the translation model
translator = pipeline("translation_en_to_es", model="Helsinki-NLP/opus-mt-en-es")

Device set to use cpu


In [159]:
# 3.Set max_length between 25-30 as suggested by the hint
translation_output = translator(first_review, max_length=27)
translated_review = translation_output[0]['translation_text']

with open("data/reference_translations.txt", 'r') as file:
    lines = file.readlines()
    translation_references = [line.strip() for line in lines]

bleu_metric = evaluate.load("bleu")
# FIX: Storing the entire dictionary result directly into bleu_score
bleu_score = bleu_metric.compute(predictions=[translated_review], references=[translation_references])

Your input_length: 365 is bigger than 0.9 * max_length: 27. You might consider increasing your max_length manually, e.g. translator('...', max_length=400)


In [160]:
# Print results to verify
print("Translated Review:", translated_review)
print("BLEU Score:", bleu_score)

Translated Review: Estoy muy satisfecho con mi 2014 Nissan NV SL. Uso esta furgoneta para mis entregas de negocios y uso personal.
BLEU Score: {'bleu': 0.6022774485691839, 'precisions': [0.9090909090909091, 0.7142857142857143, 0.55, 0.3684210526315789], 'brevity_penalty': 1.0, 'length_ratio': 1.0476190476190477, 'translation_length': 22, 'reference_length': 21}


**TASK 3: Ask Question about Car **Review****

In [161]:
from transformers import pipeline

# 1. Store the 2nd review as the context (index 1)
context = df['Review'].iloc[1]

In [162]:
# 2. Formulate the required question
question = "What did he like about the brand?"

In [163]:
# 3. Load the extractive QA pipeline with the requested model
qa_predictor = pipeline("question-answering", model="deepset/minilm-uncased-squad2")

Some weights of the model checkpoint at deepset/minilm-uncased-squad2 were not used when initializing BertForQuestionAnswering: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForQuestionAnswering from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForQuestionAnswering from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cpu


In [164]:
# 4. Run the model to find the answer
qa_result = qa_predictor(question=question, context=context)

In [165]:
# 5. Extract and store the actual text answer string
answer = qa_result['answer']

# Print results to verify
print("Question:", question)
print("Context:", context)
print("Answer:", answer)

Question: What did he like about the brand?
Context: The car is fine. It's a bit loud and not very powerful. On one hand, compared to its peers, the interior is well-built. The transmission failed a few years ago, and the dealer replaced it under warranty with no issues. Now, about 60k miles later, the transmission is failing again. It sounds like a truck, and the issues are well-documented. The dealer tells me it is normal, refusing to do anything to resolve the issue. After owning the car for 4 years, there are many other vehicles I would purchase over this one. Initially, I really liked what the brand is about: ride quality, reliability, etc. But I will not purchase another one. Despite these concerns, I must say, the level of comfort in the car has always been satisfactory, but not worth the rest of issues found.
Answer: ride quality, reliability


**TASK 4: Summarize and Analyze a car **review****

In [166]:
# 1. Grab the last review in the dataset (index -1)
last_review = df['Review'].iloc[-1]

In [167]:
# 2. Load the pre-trained summarization pipeline with the correct model path
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")

Device set to use cpu


In [168]:
# 3. Generate summary aiming for approx 50-55 tokens
summary_output = summarizer(last_review, max_length=55, min_length=50, do_sample=False)
summarized_text = summary_output[0]['summary_text']

In [169]:
# 4. Load and calculate the toxicity metric using a 1-element list
toxicity_metric = evaluate.load("toxicity", sample_ratio=1.0)
toxicity_results = toxicity_metric.compute(predictions=[summarized_text])
maximum_toxicity = max(toxicity_results['toxicity'])

Device set to use cpu


In [170]:
# 5. Load and calculate the regard metric using a 1-element list
regard_metric = evaluate.load("regard")
regard_results = regard_metric.compute(data=[summarized_text])
regard = regard_results['regard']

Device set to use cpu


In [171]:
# Print results to verify
print("Summarized Text:\n", summarized_text)
print("\nMaximum Toxicity Score:", maximum_toxicity)
print("Regard Scores:", regard)

Summarized Text:
 The Nissan Rogue provides me with the desired SUV experience without burdening me with an exorbitant payment. Handling and styling are great; I have hauled 12 bags of mulch in the back with the seats down and could have held more. The engine delivers strong

Maximum Toxicity Score: 0.00013863427739124745
Regard Scores: [[{'label': 'positive', 'score': 0.6263338923454285}, {'label': 'neutral', 'score': 0.20273476839065552}, {'label': 'other', 'score': 0.12291575223207474}, {'label': 'negative', 'score': 0.04801557958126068}]]
